# Card upgrade engine: run report
Run `python -m reco ingest && python -m reco train` first.

In [ ]:
import os, sys
from pathlib import Path

sys.path.insert(0, "..")
import duckdb, numpy as np, pandas as pd, matplotlib.pyplot as plt
from reco import bandit, features, segment

DATA = Path(os.environ.get("RECO_DATA", "../data"))
WH = DATA / "warehouse"
scores = pd.read_parquet(DATA / "user_scores.parquet")

## MCC coverage: which IBM MCCs land in which bucket, and which the `dim_mcc` join drops

In [ ]:
duckdb.sql(f"""
SELECT mcc, any_value(description) AS description, {features.BUCKET} AS bucket,
       count(*) AS n, round(sum(greatest(amount, 0))) AS spend, any_value(description) IS NULL AS dropped
FROM read_parquet('{WH}/transactions/**/*.parquet') LEFT JOIN '{WH}/dim_mcc.parquet' USING (mcc)
GROUP BY mcc ORDER BY bucket, spend DESC
""").df()

## Segmentation: silhouette by k, PCA loadings, cluster profiles

In [ ]:
_, embed, _, sil = segment.segment(pd.read_parquet(WH / "user_features.parquet"))
pd.Series(sil).plot(marker="o", title="silhouette by k", xlabel="k");

In [ ]:
pca = embed[-1]
print("explained variance:", pca.explained_variance_ratio_.round(3))
pd.DataFrame(pca.components_.T, index=features.FEATURES, columns=[f"PC{i + 1}" for i in range(pca.n_components_)]).round(2)

In [ ]:
scores.groupby("cluster")[features.FEATURES].mean().round(3).assign(users=scores.cluster.value_counts())

## Uplift: X-Learner CATE vs simulated true effect

In [ ]:
print("corr(CATE, true tau):", round(np.corrcoef(scores.cate, scores.tau)[0, 1], 3))
print("persuadable share (CATE > 0):", round((scores.cate > 0).mean(), 3))
for c, g in scores.groupby("cluster"):
    plt.scatter(g.tau, g.cate, s=5, label=c)
plt.xlabel("true tau"); plt.ylabel("CATE"); plt.legend();

## Bandit: reward vs random policy and vs oracle (regret)

In [ ]:
cashback = np.where(scores.cluster == "baseline", 0.3, 0.05) * 0.3
expected = np.column_stack([scores.base + scores.tau, scores.base + cashback, scores.base])
expected[scores.cluster == "revolver", :2] = -np.inf
oracle = expected.max(axis=1).mean()
res = pd.DataFrame({n: bandit.simulate(scores, n=n) for n in (500, 2000, 5000)}).T
res.assign(oracle=oracle, regret_per_step=oracle - res.bandit)